In [2]:
import pandas
import requests
import zipfile
import io
from pathlib import Path

In [12]:
base_dir = Path.cwd().parent  
# Path(__file__).resolve().parent in .py


downloading, extracting and savingn MEF income data

In [13]:
# Only the year changes in the MEF download link, so we keep one template
# and fill in the year with .format(year=...). No long URL is repeated.
MEF_URL_TEMPLATE = (
    "https://www1.finanze.gov.it/finanze/analisi_stat/public/v_4_0_0/contenuti/"
    "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_{year}.zip?d=1615465800"
)
# Years as integers, not strings: later we do maths with them (2024 - 2014)
# and merge on a numeric "year" column.
INCOME_YEARS = [2014, 2024]

mef_raw_data_dir = base_dir / "data" / "raw" / "mef"

# Create the 'data/raw/mef' directory if it doesn't already exist
mef_raw_data_dir.mkdir(parents=True, exist_ok=True)

# File types we accept from inside the ZIP (a tuple, because endswith() accepts tuples)
extensions = (".csv", ".xlsx", ".xls")

for year in INCOME_YEARS:
    url = MEF_URL_TEMPLATE.format(year=year)

    # Cache: if a file for this year is already on disk, skip the download.
    # mef_raw_data_dir / "..." is already a Path, so no need to wrap it in Path().
    if any((mef_raw_data_dir / f"mef_income_{year}{ext}").exists() for ext in extensions):
        print(f"mef_income_{year} already downloaded, skipping.")
        continue

    print(f"Downloading {year} from {url} ...")
    # timeout=30: give up after 30 seconds instead of freezing forever
    response = requests.get(url, timeout=30)
    # turn HTTP errors (404, 500, ...) into exceptions
    response.raise_for_status()

    # Open the ZIP directly from memory (no temporary .zip file on disk)
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        # .lower() so that "DATA.CSV" is accepted too, not only "data.csv"
        data_files = [
            info for info in z.infolist()
            if info.filename.lower().endswith(extensions)
        ]

        # We save every file as "mef_income_<year>.<ext>". With two data files
        # in one ZIP, the second would overwrite the first, so we stop and say so.
        if len(data_files) != 1:
            raise ValueError(
                f"Expected 1 data file in the {year} ZIP, found {len(data_files)}: "
                f"{[info.filename for info in data_files]}"
            )

        fileinfo = data_files[0]
        extension = Path(fileinfo.filename).suffix.lower()
        target_path = mef_raw_data_dir / f"mef_income_{year}{extension}"

        # Read the file out of the ZIP and write it to disk
        target_path.write_bytes(z.read(fileinfo.filename))
        print(f"Extracted to: {target_path}")

Extracted to: /Users/armanfayyazi/Desktop/Folders/unimi/coding/project/housing-affordability-italy/data/raw/mef/mef_income_2014.csv
Extracted to: /Users/armanfayyazi/Desktop/Folders/unimi/coding/project/housing-affordability-italy/data/raw/mef/mef_income_2024.csv


In [ ]:
_